In [ ]:
import pandas as pd
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import statistics as st

pd.options.display.max_rows = 200

In [ ]:
df = pd.DataFrame()
for i in range(2,5):
    file_name = f"./round-{i}-island-data-bottle/prices_round_{i}_day_1.csv"
    df_day_i = pd.read_csv(file_name, sep=';')
    df = pd.concat([df, df_day_i])
    
df = df.sort_values(by='timestamp', ascending=True).reset_index()

In [ ]:
file_name = f"./round-{2}-island-data-bottle/prices_round_{2}_day_1.csv"
df_day_2 = pd.read_csv(file_name, sep=';')

In [ ]:
df_day_2.columns

In [ ]:
file_name = f"./round-{3}-island-data-bottle/prices_round_{3}_day_1.csv"
df_day_3 = pd.read_csv(file_name, sep=';')

In [ ]:
df_day_3.columns

In [ ]:
file_name = f"./round-{4}-island-data-bottle/prices_round_{4}_day_1.csv"
df_day_4 = pd.read_csv(file_name, sep=';')

In [ ]:
df_day_4

In [ ]:
df_day_3_pivoted = df_day_3.pivot(index='timestamp', columns='product', values='mid_price')
df_day_4_pivoted = df_day_4.pivot(index='timestamp', columns='product', values='mid_price')

In [ ]:
df_day_3_pivoted.reset_index(inplace=True)
df_day_4_pivoted.reset_index(inplace=True)

In [ ]:
df = pd.merge(df_day_2, df_day_3_pivoted, on='timestamp', how='outer')
df = pd.merge(df, df_day_4_pivoted, on='timestamp', how='outer')

In [ ]:
columns_order = ['timestamp', 'ORCHIDS', 'TRANSPORT_FEES', 'EXPORT_TARIFF', 'IMPORT_TARIFF', 'SUNLIGHT', 'HUMIDITY', 'DAY']
product_columns = [col for col in df.columns if col not in columns_order]
columns_order.extend(product_columns)
df = df[columns_order]

In [ ]:
df.to_csv('day_1_consolidated.csv', index=False)

In [ ]:
df = pd.read_csv('day_1_consolidated.csv')

In [ ]:
df

In [ ]:
df.columns

# helpers

In [ ]:
def get_prev_returns(df, col, its):
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_from_{its}_its_ago"] = (df[col] - df[prev_col]) / df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    return df

def get_future_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    df[f"{col}_returns_in_{its}_its"] = (df[future_col] - df[col]) / df[col]
    df.drop(columns=[future_col], inplace=True)
    return df

def get_centered_returns(df, col, its):
    future_col = f"{col}_future_{its}_its"
    df[future_col] = df[col].shift(-its)
    prev_col = f"{col}_prev_{its}_its"
    df[prev_col] = df[col].shift(its)
    df[f"{col}_returns_centered_with_{its}_its"] = (df[future_col] - df[prev_col])/df[prev_col]
    df.drop(columns=[prev_col], inplace=True)
    df.drop(columns=[future_col], inplace=True)
    return df

# regs

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500]
symbols = ['ORCHIDS', 'CHOCOLATE', 'GIFT_BASKET', 'ROSES', 'STRAWBERRIES', 'COCONUT', 'COCONUT_COUPON']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from other symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and not col.startswith(target_symbol)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + symbols]
    
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500]
symbols = ['ORCHIDS','GIFT_BASKET']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from other symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and not col.startswith(target_symbol)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + symbols]
    
    print()

In [ ]:
df

In [ ]:
import plotly.graph_objects as go

# Create traces for ORCHIDS and GIFT_BASKET prices
trace_orchids = go.Scatter(
    x=df['timestamp'],
    y=df['ORCHIDS'],
    mode='lines',
    name='ORCHIDS',
    yaxis='y1'  # Assign the trace to the first y-axis
)

trace_gift_basket = go.Scatter(
    x=df['timestamp'],
    y=df['GIFT_BASKET'],
    mode='lines',
    name='GIFT_BASKET',
    yaxis='y2'  # Assign the trace to the second y-axis
)

# Create the layout for the plot
layout = go.Layout(
    title='ORCHIDS and GIFT_BASKET Prices Over Time',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(
        title='ORCHIDS Price',
        titlefont=dict(color='blue'),
        tickfont=dict(color='blue')
    ),
    yaxis2=dict(
        title='GIFT_BASKET Price',
        titlefont=dict(color='green'),
        tickfont=dict(color='green'),
        overlaying='y',
        side='right'
    )
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[trace_orchids, trace_gift_basket], layout=layout)

# Display the plot
fig.show()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500]
symbols = ['ORCHIDS','GIFT_BASKET']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from other symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and not col.startswith(target_symbol)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_from_{timeframe}_its_ago"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + symbols]
    
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

timeframes = [1, 2, 3, 5, 7, 10, 15, 25, 50, 75, 100, 150, 200, 250, 300, 350, 400, 500, 600, 700, 800, 900, 1000]
predictor_symbols = ['GIFT_BASKET']
responder_symbols = ['ORCHIDS']

df_copy = df.copy()

for timeframe in timeframes:
    print(f"Timeframe: {timeframe} iterations")
    
    # Add lagged and future returns columns for each symbol
    for symbol in predictor_symbols + responder_symbols:
        df_copy = get_prev_returns(df_copy, symbol, timeframe)
        df_copy = get_future_returns(df_copy, symbol, timeframe)
    
    for target_symbol in responder_symbols:
        print(f"Target Symbol: {target_symbol}")
        
        # Get the feature columns (lagged returns from predictor symbols)
        feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
        
        # Get the target column (future returns for the target symbol)
        target_col = f"{target_symbol}_returns_in_{timeframe}_its"
        
        # Drop rows with missing values
        df_train = df_copy[feature_cols + [target_col]].dropna()
        
        # Split the data into features (X) and target (y)
        X = df_train[feature_cols]
        y = df_train[target_col]
        
        # Create and fit the linear regression model (without y-intercept)
        model = LinearRegression(fit_intercept=False)
        model.fit(X, y)
        
        # Print the learned equation
        equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
        print("Learned Equation:")
        print(equation)
        
        # Make predictions on the training data
        y_pred = model.predict(X)
        
        # Calculate and print the R-squared and p-value
        r2 = r2_score(y, y_pred)
        print(f"R-squared: {r2:.4f}")
        
        _, p_value = stats.pearsonr(y, y_pred)
        print(f"p-value: {p_value:.4f}")
        
        print()
    
    df_copy = df_copy[['timestamp'] + predictor_symbols + responder_symbols]
    print()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
import scipy.stats as stats

predictor_timeframes = [50, 75, 100, 150, 200, 250, 300, 350, 400, 500]
responder_timeframes = [50, 75, 100, 150, 200, 250, 300, 350, 400, 500]
predictor_symbols = ['GIFT_BASKET']
responder_symbols = ['ORCHIDS']

df_copy = df.copy()

for responder_timeframe in responder_timeframes:
    print(f"Responder Timeframe: {responder_timeframe} iterations")
    
    # Add future returns columns for responder symbols
    for symbol in responder_symbols:
        df_copy = get_future_returns(df_copy, symbol, responder_timeframe)
    
    for predictor_timeframe in predictor_timeframes:
        print(f"Predictor Timeframe: {predictor_timeframe} iterations")
        
        # Add lagged returns columns for predictor symbols
        for symbol in predictor_symbols:
            df_copy = get_prev_returns(df_copy, symbol, predictor_timeframe)
        
        for target_symbol in responder_symbols:
            print(f"Target Symbol: {target_symbol}")
            
            # Get the feature columns (lagged returns from predictor symbols)
            feature_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{predictor_timeframe}_its_ago") and any(col.startswith(symbol) for symbol in predictor_symbols)]
            
            # Get the target column (future returns for the target symbol)
            target_col = f"{target_symbol}_returns_in_{responder_timeframe}_its"
            
            # Drop rows with missing values
            df_train = df_copy[feature_cols + [target_col]].dropna()
            
            # Split the data into features (X) and target (y)
            X = df_train[feature_cols]
            y = df_train[target_col]
            
            # Create and fit the linear regression model (without y-intercept)
            model = LinearRegression(fit_intercept=False)
            model.fit(X, y)
            
            # Print the learned equation
            equation = f"{target_col} = " + " + ".join([f"{coef:.4f} * {feat}" for feat, coef in zip(feature_cols, model.coef_)])
            print("Learned Equation:")
            print(equation)
            
            # Make predictions on the training data
            y_pred = model.predict(X)
            
            # Calculate and print the R-squared and p-value
            r2 = r2_score(y, y_pred)
            print(f"R-squared: {r2:.4f}")
            
            _, p_value = stats.pearsonr(y, y_pred)
            print(f"p-value: {p_value:.4f}")
            
            print()
        
        # Remove lagged returns columns for predictor symbols
        lagged_cols = [col for col in df_copy.columns if col.endswith(f"_returns_from_{predictor_timeframe}_its_ago")]
        df_copy.drop(columns=lagged_cols, inplace=True)
    
    # Remove future returns columns for responder symbols
    future_cols = [col for col in df_copy.columns if col.endswith(f"_returns_in_{responder_timeframe}_its")]
    df_copy.drop(columns=future_cols, inplace=True)
    
    print()

# backtest

learned eq: 
ORCHIDS_returns_in_500_its = -5.2917 * GIFT_BASKET_returns_from_500_its_ago
R-squared: 0.2920

In [ ]:
df

In [ ]:
df_backtest = df.copy()[['timestamp','ORCHIDS', 'GIFT_BASKET']]

In [ ]:
df_backtest = get_prev_returns(df_backtest, 'GIFT_BASKET', 500)
df_backtest = get_future_returns(df_backtest, 'ORCHIDS', 500)

In [ ]:
df_backtest = df_backtest.dropna()

In [ ]:
df_backtest['ORCHIDS_pred_returns_in_500_its'] = df_backtest['GIFT_BASKET_returns_from_500_its_ago'] * -5.2917

In [ ]:
take_threshold = -0.015
clear_threshold = -0.0005

In [ ]:

df_backtest['orchids_signal'] = df_backtest.apply(lambda row: "SHORT" if row['ORCHIDS_pred_returns_in_500_its'] < take_threshold else "CLEAR" if row['ORCHIDS_pred_returns_in_500_its'] > clear_threshold else None,  axis=1)

In [ ]:
df_backtest['last_orchids_signal'] = df_backtest['orchids_signal'].fillna(method='ffill')

In [ ]:
df_backtest.loc[df_backtest['last_orchids_signal'] == 'CLEAR', 'orchids_target_position'] = 0
df_backtest.loc[df_backtest['last_orchids_signal'] == 'SHORT', 'orchids_target_position'] = -100

In [ ]:
df_backtest['orchids_target_position'] = df_backtest['orchids_target_position'].fillna(0)

In [ ]:
df_backtest['orchids_target_position_change'] = df_backtest['orchids_target_position'].diff(1)

In [ ]:
df_backtest['cash'] = -df_backtest['orchids_target_position_change'] * df_backtest['ORCHIDS']

In [ ]:
df_backtest['cumulative_cash'] = df_backtest['cash'].cumsum()

In [ ]:
df_backtest['position_mark'] = df_backtest['orchids_target_position'] * df_backtest['ORCHIDS']

In [ ]:
df_backtest['pnl'] = df_backtest['position_mark'] + df_backtest['cumulative_cash']

In [ ]:
df_backtest['pnl'].tail(3)

In [ ]:

df_backtest.to_csv('backtest.csv', index=False)

In [ ]:
import plotly.graph_objects as go

# Create the first trace for ORCHIDS_pred_returns_in_500_its
trace1 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ORCHIDS_pred_returns_in_500_its'],
    name='ORCHIDS Predicted Returns',
    yaxis='y1'
)

# Create the second trace for pnl
trace2 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['pnl'],
    name='PnL',
    yaxis='y2'
)

# Create the third trace for ORCHIDS
trace3 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ORCHIDS'],
    name='ORCHIDS',
    yaxis='y3'
)

# Create the layout for the graph
layout = go.Layout(
    title='ORCHIDS, Predicted Returns, and PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(
        title='ORCHIDS Predicted Returns',
        side='left'
    ),
    yaxis2=dict(
        title='PnL',
        side='right',
        overlaying='y'
    ),
    yaxis3=dict(
        title='ORCHIDS',
        side='right',
        overlaying='y',
        position=0.95
    )
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[trace1, trace2, trace3], layout=layout)

# Show the plot
fig.show()

In [ ]:
0.02*1000

In [ ]:
20 - 5

In [ ]:
import io

# backtest on other days 

In [ ]:
def _process_data_(file):
    with open(file, 'r') as file:
        log_content = file.read()
    sections = log_content.split('Sandbox logs:')[1].split('Activities log:')
    sandbox_log =  sections[0].strip()
    activities_log = sections[1].split('Trade History:')[0]
    # sandbox_log_list = [json.loads(line) for line in sandbox_log.split('\n')]
    trade_history =  json.loads(sections[1].split('Trade History:')[1])
    # sandbox_log_df = pd.DataFrame(sandbox_log_list)
    market_data_df = pd.read_csv(io.StringIO(activities_log), sep=";", header=0)
    trade_history_df = pd.json_normalize(trade_history)
    return market_data_df, trade_history_df

In [ ]:
df3, _ = _process_data_('./round-4-island-data-bottle/round_3_results.log')

In [ ]:
df_day_3

In [ ]:
file_name = f"./round-{3}-island-data-bottle/prices_round_{3}_day_0.csv"
df_round_3 = pd.read_csv(file_name, sep=';')

file_name = f"./round-{2}-island-data-bottle/prices_round_{2}_day_0.csv"
df_round_2 = pd.read_csv(file_name, sep=';')

In [ ]:
df_round_2 = df_round_2[['timestamp', 'ORCHIDS']]

In [ ]:
df_round_2

In [ ]:
df_round_3 = df_round_3.pivot(columns='product', index='timestamp', values='mid_price').reset_index()

In [ ]:
df = df_round_2.merge(df_round_3, on='timestamp', how='inner')

In [ ]:
df_backtest = df.copy()[['timestamp','ORCHIDS', 'GIFT_BASKET']]

In [ ]:
df_backtest = get_prev_returns(df_backtest, 'GIFT_BASKET', 500)
df_backtest = get_future_returns(df_backtest, 'ORCHIDS', 500)

In [ ]:
df_backtest = df_backtest.dropna()

In [ ]:
df_backtest['ORCHIDS_pred_returns_in_500_its'] = df_backtest['GIFT_BASKET_returns_from_500_its_ago'] * -5.2917

In [ ]:
df_backtest.describe()

In [ ]:
df_backtest

In [ ]:
take_threshold = -0.015
clear_threshold = -0.0005

In [ ]:

df_backtest['orchids_signal'] = df_backtest.apply(lambda row: "SHORT" if row['ORCHIDS_pred_returns_in_500_its'] < take_threshold else "CLEAR" if row['ORCHIDS_pred_returns_in_500_its'] > clear_threshold else None,  axis=1)

In [ ]:
df_backtest

In [ ]:
df_backtest['orchids_signal'].value_counts()

In [ ]:
df_backtest['last_orchids_signal'] = df_backtest['orchids_signal'].fillna(method='ffill')

In [ ]:
df_backtest['last_orchids_signal'].value_counts()

In [ ]:
df_backtest.loc[df_backtest['last_orchids_signal'] == 'CLEAR', 'orchids_target_position'] = 0
df_backtest.loc[df_backtest['last_orchids_signal'] == 'SHORT', 'orchids_target_position'] = -100

In [ ]:
df_backtest['orchids_target_position'] = df_backtest['orchids_target_position'].fillna(0)

In [ ]:
df_backtest

In [ ]:
df_backtest['orchids_target_position_change'] = df_backtest['orchids_target_position'].diff(1)

In [ ]:
df_backtest['cash'] = -df_backtest['orchids_target_position_change'] * df_backtest['ORCHIDS']

In [ ]:
df_backtest['cumulative_cash'] = df_backtest['cash'].cumsum()

In [ ]:
df_backtest['position_mark'] = df_backtest['orchids_target_position'] * df_backtest['ORCHIDS']

In [ ]:
df_backtest['pnl'] = df_backtest['position_mark'] + df_backtest['cumulative_cash']

In [ ]:
df_backtest['pnl'].tail(3)

In [ ]:
df

In [ ]:
import plotly.graph_objects as go

# Create the first trace for ORCHIDS_pred_returns_in_500_its
trace1 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ORCHIDS_pred_returns_in_500_its'],
    name='ORCHIDS Predicted Returns',
    yaxis='y1'
)

# Create the second trace for pnl
trace2 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['pnl'],
    name='PnL',
    yaxis='y2'
)

# Create the third trace for ORCHIDS
trace3 = go.Scatter(
    x=df_backtest['timestamp'],
    y=df_backtest['ORCHIDS'],
    name='ORCHIDS',
    yaxis='y3'
)

# Create the layout for the graph
layout = go.Layout(
    title='ORCHIDS, Predicted Returns, and PnL',
    xaxis=dict(title='Timestamp'),
    yaxis=dict(
        title='ORCHIDS Predicted Returns',
        side='left'
    ),
    yaxis2=dict(
        title='PnL',
        side='right',
        overlaying='y'
    ),
    yaxis3=dict(
        title='ORCHIDS',
        side='right',
        overlaying='y',
        position=0.95
    )
)

# Create the figure and add the traces and layout
fig = go.Figure(data=[trace1, trace2, trace3], layout=layout)

# Show the plot
fig.show()